In [1]:
import numpy as np
import pandas as pd

TEST_PATH = "/kaggle/input/datasets/liaawies/submission-sample/new_submission_sample (3) (1).csv"
BERT_TEST_PATH = "/kaggle/input/datasets/liaawies/test-probs-bert/test_bert_probs.npy" 
sub = pd.read_csv(TEST_PATH)
test_probs = np.load(BERT_TEST_PATH)

sub['target'] = test_probs
sub.to_csv("submission_bert_baseline.csv", index=False)

print("Страховочный сабмит готов: submission_bert_baseline.csv")
print(f"Размерность: {sub.shape}, Средняя вероятность: {sub['target'].mean():.4f}")

Страховочный сабмит готов: submission_bert_baseline.csv
Размерность: (11000, 4), Средняя вероятность: 0.4784


In [3]:
df_bert = pd.read_csv("submission_bert_baseline.csv")

if "Unnamed: 0" in df_bert.columns:
    row_id = df_bert["Unnamed: 0"]
elif "row_id" in df_bert.columns:
    row_id = df_bert["row_id"]
else:
    row_id = df_bert.index

sub_fixed = pd.DataFrame({
    "row_id": row_id,
    "target": df_bert["target"]
})

sub_fixed.to_csv("submission_bert_fixed.csv", index=False)

print("Файл submission_bert_fixed.csv готов")
print(f"Колонки: {sub_fixed.columns.tolist()}")
print(sub_fixed.head())

Файл submission_bert_fixed.csv готов
Колонки: ['row_id', 'target']
   row_id    target
0       0  0.001305
1       1  0.060529
2       2  0.988755
3       3  0.992184
4       4  0.000392


In [4]:
df_bert = pd.read_csv("submission_bert_fixed.csv")
df_lgb = pd.read_csv("/kaggle/input/datasets/liaawies/lgb-final/submission_lgb_FINAL.csv")

w_bert = 0.55
w_lgb = 0.45

blend_probs = w_bert * df_bert["target"] + w_lgb * df_lgb["target"]

sub_ensemble_2m = pd.DataFrame({
    "row_id": df_lgb["row_id"],
    "target": blend_probs
})

sub_ensemble_2m.to_csv("submission_ensemble_bert_lgb.csv", index=False)

print("Файл submission_ensemble_bert_lgb.csv готов")
print(f"Размерность: {sub_ensemble_2m.shape}")
print(f"Первые 5 строк:\n{sub_ensemble_2m.head()}")

Файл submission_ensemble_bert_lgb.csv готов
Размерность: (11000, 2)
Первые 5 строк:
   row_id    target
0       0  0.020514
1       1  0.114758
2       2  0.993777
3       3  0.975431
4       4  0.002378


In [5]:
import pickle

with open('/kaggle/input/datasets/liaawies/feach-cols/feature_cols.pkl', 'rb') as f:
    cb_features = pickle.load(f)

print(f"Всего фич у CatBoost: {len(cb_features)}")
print("\nСписок колонок:")
print(cb_features)

Всего фич у CatBoost: 17

Список колонок:
['text_length', 'word_count', 'unique_ratio', 'upper_ratio', 'has_parentheses', 'digit_count', 'ru_ratio', 'en_ratio', 'has_bed_config', 'has_room_level', 'has_view', 'has_balcony', 'is_family', 'business_completeness', 'hotel_positive_rate', 'hotel_room_count', 'hotel_target_std']


In [6]:
import catboost as cb

TEST_PATH = "/kaggle/input/datasets/liaawies/submission-sample/new_submission_sample (3) (1).csv"
PUBLIC_TRAIN_PATH = "/kaggle/input/datasets/liaawies/dataset2/public_dataset.csv"
MODEL_CB_PATH = "/kaggle/input/models/liaawies/model-cb/other/default/1/model_cb.pkl"
FEATURE_COLS_PATH = "/kaggle/input/datasets/liaawies/feach-cols/feature_cols.pkl"

print("Загрузка тестового датасета и модели")
test_df = pd.read_csv(TEST_PATH)
train_df = pd.read_csv(PUBLIC_TRAIN_PATH)

with open(FEATURE_COLS_PATH, 'rb') as f:
    cb_features = pickle.load(f)

with open(MODEL_CB_PATH, 'rb') as f:
    cb_model = pickle.load(f)

print("Генерация фич для CatBoost")

# Приводим hotel_id к int
test_df['hotel_id'] = pd.to_numeric(test_df['hotel_id'], errors='coerce').fillna(-1).astype(int)
train_df['hotel_id'] = pd.to_numeric(train_df['hotel_id'], errors='coerce').fillna(-1).astype(int)

# Нормализация текста
def clean_text(x):
    if pd.isna(x):
        return ""
    return str(x).strip()

texts = test_df['supplier_room_name'].map(clean_text)
test_df['text_clean'] = texts
test_df['supplier_room_name'] = texts

# Текстовые и символьные фичи
test_df['text_length'] = texts.str.len()
test_df['word_count'] = texts.apply(lambda x: len(x.split()))
test_df['unique_ratio'] = texts.apply(lambda x: len(set(x.split())) / max(len(x.split()), 1))
test_df['upper_ratio'] = texts.apply(lambda x: sum(1 for c in x if c.isupper()) / max(len(x), 1))
test_df['has_parentheses'] = texts.str.contains(r'[\(\)]', regex=True).astype(int)
test_df['digit_count'] = texts.apply(lambda x: sum(1 for c in x if c.isdigit()))

# Языковые пропорции
def get_char_ratios(text):
    if not text:
        return 0.0, 0.0
    ru_cnt = sum(1 for c in text.lower() if 'а' <= c <= 'я' or c == 'ё')
    en_cnt = sum(1 for c in text.lower() if 'a' <= c <= 'z')
    tot = max(len(text), 1)
    return ru_cnt / tot, en_cnt / tot

ratios = texts.apply(get_char_ratios)
test_df['ru_ratio'] = [r[0] for r in ratios]
test_df['en_ratio'] = [r[1] for r in ratios]

# Бизнес-флаги
lower_texts = texts.str.lower()
test_df['has_bed_config'] = lower_texts.str.contains(r'king|queen|twin|bed|кровать|двуспальная|односпальная', regex=True).astype(int)
test_df['has_room_level'] = lower_texts.str.contains(r'deluxe|suite|standard|superior|lux|люкс|стандарт|улучшенный', regex=True).astype(int)
test_df['has_view'] = lower_texts.str.contains(r'view|вид|sea|ocean|marina|city|park|mountain', regex=True).astype(int)
test_df['has_balcony'] = lower_texts.str.contains(r'balcony|terrace|балкон|терраса', regex=True).astype(int)
test_df['is_family'] = lower_texts.str.contains(r'family|quadruple|triple|семейный|детская', regex=True).astype(int)

# Полнота описания
test_df['business_completeness'] = (
    test_df['has_bed_config'] + 
    test_df['has_room_level'] + 
    test_df['has_view'] + 
    test_df['has_balcony'] + 
    test_df['is_family']
)

# Агрегаты отелей
hotel_stats = train_df.groupby('hotel_id').agg(
    hotel_positive_rate=('target', 'mean'),
    hotel_room_count=('target', 'count'),
    hotel_target_std=('target', 'std')
).reset_index()

test_df = test_df.merge(hotel_stats, on='hotel_id', how='left')

test_df['hotel_positive_rate'] = test_df['hotel_positive_rate'].fillna(train_df['target'].mean())
test_df['hotel_room_count'] = test_df['hotel_room_count'].fillna(1)
test_df['hotel_target_std'] = test_df['hotel_target_std'].fillna(0.0)

print("Расчет вероятностей CatBoost")

if hasattr(cb_model, 'feature_names_') and cb_model.feature_names_:
    model_features = list(cb_model.feature_names_)
else:
    model_features = list(cb_features)

print(f"Признаки модели ({len(model_features)}): {model_features}")

X_test = test_df[model_features].copy()

text_features_list = []
if hasattr(cb_model, 'get_param'):
    text_features_list = cb_model.get_param('text_features') or []

possible_text_cols = ['supplier_room_name', 'text_clean']
for col in possible_text_cols:
    if col in X_test.columns and col not in text_features_list:
        text_features_list.append(col)

for col in X_test.columns:
    if col in text_features_list or X_test[col].dtype == 'object':
        X_test[col] = X_test[col].astype(str)
        if col not in text_features_list:
            text_features_list.append(col)
    else:
        X_test[col] = pd.to_numeric(X_test[col], errors='coerce').fillna(0).astype(float)

print(f"Текстовые признаки, переданные в CatBoost: {text_features_list}")

test_pool = cb.Pool(X_test, text_features=text_features_list)

cb_probs = cb_model.predict_proba(test_pool)[:, 1]

if "row_id" in test_df.columns:
    row_id = test_df["row_id"]
elif "Unnamed: 0" in test_df.columns:
    row_id = test_df["Unnamed: 0"]
else:
    row_id = test_df.index

sub_cb = pd.DataFrame({
    "row_id": row_id,
    "target": cb_probs
})

sub_cb.to_csv("submission_cb_only.csv", index=False)

print("\n Файл submission_cb_only.csv готов")
print(f"Размерность: {sub_cb.shape}")
print(f"Среднее предсказанное значение: {cb_probs.mean():.4f}")
print(sub_cb.head())

Загрузка тестового датасета и модели
Генерация фич для CatBoost
Расчет вероятностей CatBoost
Признаки модели (18): ['text_length', 'word_count', 'unique_ratio', 'upper_ratio', 'has_parentheses', 'digit_count', 'ru_ratio', 'en_ratio', 'has_bed_config', 'has_room_level', 'has_view', 'has_balcony', 'is_family', 'business_completeness', 'hotel_positive_rate', 'hotel_room_count', 'hotel_target_std', 'text_clean']
Текстовые признаки, переданные в CatBoost: ['text_clean']

 Файл submission_cb_only.csv готов
Размерность: (11000, 2)
Среднее предсказанное значение: 0.3808
   row_id    target
0       0  0.186011
1       1  0.190243
2       2  0.998349
3       3  0.356069
4       4  0.054686


In [7]:
from scipy.stats import rankdata

df_bert = pd.read_csv("submission_bert_fixed.csv")
df_lgb = pd.read_csv("/kaggle/input/datasets/liaawies/lgb-final/submission_lgb_FINAL.csv")
df_cb = pd.read_csv("submission_cb_only.csv")

rank_bert = rankdata(df_bert['target']) / len(df_bert)
rank_lgb = rankdata(df_lgb['target']) / len(df_lgb)
rank_cb = rankdata(df_cb['target']) / len(df_cb)

final_rank_blend = 0.80 * rank_bert + 0.15 * rank_lgb + 0.05 * rank_cb

sub = pd.DataFrame({
    'row_id': df_bert['row_id'],
    'target': final_rank_blend
})
sub.to_csv("submission_rank_blend.csv", index=False)

In [8]:
import os

cb_path = "/kaggle/input/datasets/liaawies/sub-stuck/submission_cb_stacked.csv"
lgb_path = "/kaggle/input/datasets/liaawies/lgb-final/submission_lgb_FINAL.csv"

df_cb = pd.read_csv(cb_path)
df_lgb = pd.read_csv(lgb_path)

print(f"CatBoost shape: {df_cb.shape}")
print(f"LightGBM shape: {df_lgb.shape}")

if not df_cb["row_id"].equals(df_lgb["row_id"]):
    print("row_id не совпадают! Выравниваем по row_id...")
    df_lgb_aligned = df_lgb.set_index("row_id").reindex(df_cb["row_id"]).reset_index()
    df_lgb = df_lgb_aligned
else:
    print("row_id совпадают.")

w_cb = 0.5
w_lgb = 0.5

blend_probs = w_cb * df_cb["target"] + w_lgb * df_lgb["target"]

sub_ensemble_cb_lgb = pd.DataFrame({
    "row_id": df_cb["row_id"],
    "target": blend_probs
})

output_path = "submission_ensemble_cb_lgb.csv"
sub_ensemble_cb_lgb.to_csv(output_path, index=False)

if os.path.exists(output_path):
    print(f"Файл {output_path} успешно создан!")
    print(f"Размер: {os.path.getsize(output_path)} байт")
else:
    print("Ошибка: файл не создан")

print(f"Первые 5 строк:\n{sub_ensemble_cb_lgb.head()}")

CatBoost shape: (11000, 2)
LightGBM shape: (11000, 2)
row_id совпадают.
Файл submission_ensemble_cb_lgb.csv успешно создан!
Размер: 263648 байт
Первые 5 строк:
   row_id    target
0       0  0.501607
1       1  0.586836
2       2  0.999908
3       3  0.976882
4       4  0.459609


In [9]:
df_bert = pd.read_csv("/kaggle/input/datasets/liaawies/bert-fix/submission_bert_fixed.csv")
df_lgb  = pd.read_csv("/kaggle/input/datasets/liaawies/sub-lgb/submission_lgb_only.csv")
df_cb   = pd.read_csv("/kaggle/input/datasets/liaawies/sub-stuck/submission_cb_stacked.csv") # новый CatBoost со стекингом!

rank_bert = rankdata(df_bert["target"]) / len(df_bert)
rank_lgb  = rankdata(df_lgb["target"]) / len(df_lgb)
rank_cb   = rankdata(df_cb["target"]) / len(df_cb)

blend_ranks = 0.50 * rank_bert + 0.30 * rank_cb + 0.20 * rank_lgb

sub_ensemble = pd.DataFrame({
    "row_id": df_bert["row_id"],
    "target": blend_ranks
})

sub_ensemble.to_csv("submission_ensemble_stacked.csv", index=False)
print("Итоговый сабмит submission_ensemble_stacked.csv готов")

Итоговый сабмит submission_ensemble_stacked.csv готов


In [10]:
df_bert = pd.read_csv("/kaggle/input/datasets/liaawies/bert-fix/submission_bert_fixed.csv")
df_lgb  = pd.read_csv("/kaggle/input/datasets/liaawies/sub-lgb/submission_lgb_only.csv")
df_cb   = pd.read_csv("/kaggle/input/datasets/liaawies/sub-stuck/submission_cb_stacked.csv")  

def get_power_ranks(probs, power=1.5):
    ranks = rankdata(probs) / len(probs)
    return np.power(ranks, power)

POWER = 1.5 

rank_bert = get_power_ranks(df_bert["target"], power=POWER)
rank_cb   = get_power_ranks(df_cb["target"],   power=POWER)
rank_lgb  = get_power_ranks(df_lgb["target"],  power=POWER)

w_cb   = 0.50  
w_bert = 0.35  
w_lgb  = 0.15  

blend_power_ranks = (w_cb * rank_cb) + (w_bert * rank_bert) + (w_lgb * rank_lgb)

final_target = rankdata(blend_power_ranks) / len(blend_power_ranks)

sub_power = pd.DataFrame({
    "row_id": df_cb["row_id"],
    "target": final_target
})

sub_power.to_csv("submission_power_rank_ensemble.csv", index=False)

print("Файл submission_power_rank_ensemble.csv готов")
print(f"Размерность: {sub_power.shape}")
print(f"Минимум: {final_target.min():.4f}, Максимум: {final_target.max():.4f}")
print(sub_power.head())

Файл submission_power_rank_ensemble.csv готов
Размерность: (11000, 2)
Минимум: 0.0001, Максимум: 1.0000
   row_id    target
0       0  0.200636
1       1  0.545182
2       2  0.906182
3       3  0.852273
4       4  0.092091


In [11]:
def clip_rank_outliers(ranks, low_percentile=0.005, high_percentile=0.995):
    p_low = np.percentile(ranks, low_percentile * 100)
    p_high = np.percentile(ranks, high_percentile * 100)
    
    clipped = np.clip(ranks, p_low, p_high)
    
    return rankdata(clipped) / len(clipped)

final_target_clipped = clip_rank_outliers(sub_power["target"].values)
sub_power["target"] = final_target_clipped
sub_power.to_csv("submission_power_clipped.csv", index=False)

In [12]:
df_bert = pd.read_csv("/kaggle/input/datasets/liaawies/bert-fix/submission_bert_fixed.csv")
df_lgb  = pd.read_csv("/kaggle/input/datasets/liaawies/sub-lgb/submission_lgb_only.csv")
df_cb   = pd.read_csv("/kaggle/input/datasets/liaawies/sub-stuck/submission_cb_stacked.csv")

def get_power_ranks(probs, power=1.5):
    ranks = rankdata(probs) / len(probs)
    return np.power(ranks, power)

POWER = 1.6  

r_cb   = get_power_ranks(df_cb["target"],   power=POWER)
r_bert = get_power_ranks(df_bert["target"], power=POWER)
r_lgb  = get_power_ranks(df_lgb["target"],  power=POWER)

w_cb   = 0.45
w_bert = 0.40
w_lgb  = 0.15

blend = w_cb * r_cb + w_bert * r_bert + w_lgb * r_lgb

final_ranks = rankdata(blend) / len(blend)

def clip_rank_outliers(ranks, low_p=0.002, high_p=0.998):
    p_low = np.percentile(ranks, low_p * 100)
    p_high = np.percentile(ranks, high_p * 100)
    clipped = np.clip(ranks, p_low, p_high)
    return rankdata(clipped) / len(clipped)

final_target_clipped = clip_rank_outliers(final_ranks)

sub_final = pd.DataFrame({
    "row_id": df_bert["row_id"],
    "target": final_target_clipped
})

output_path = "submission_power_rank_final.csv"
sub_final.to_csv(output_path, index=False)

print(f"Файл {output_path} готов")
print(f"Размерность: {sub_final.shape}")
print(f"Минимум: {sub_final['target'].min():.4f}, Максимум: {sub_final['target'].max():.4f}")
print("\nПервые 5 строк:")
print(sub_final.head())

Файл submission_power_rank_final.csv готов
Размерность: (11000, 2)
Минимум: 0.0010, Максимум: 0.9990

Первые 5 строк:
   row_id    target
0       0  0.194636
1       1  0.539182
2       2  0.895545
3       3  0.854182
4       4  0.085182


In [13]:
from scipy.optimize import minimize
from sklearn.metrics import average_precision_score

POWER = 1.5  

print("[1/3] Поиск валидационных данных")

# Пытаемся получить y_val
try:
    y_val = y_val.values if hasattr(y_val, 'values') else y_val
    val_cb = y_val_proba if 'y_val_proba' in locals() else val_cb
    print("y_val и val_cb найдены в памяти!")
except NameError:
    
    try:
        y_val = np.load("y_val.npy")
        val_cb = np.load("y_val_proba_cb_stacked.npy")
        print("✓ y_val и val_cb загружены из .npy файлов!")
    except FileNotFoundError:
        y_val, val_cb = None, None
        print("Валидационные данные не найдены. Будет использован фоллбэк с предустановленными весами")

if y_val is not None:
    try:
        val_bert
    except NameError:
        try:
            val_bert = np.load("val_bert_probs.npy")
        except FileNotFoundError:
            val_bert = val_cb.copy()
            print("val_bert не найден, дублируем val_cb.")

    try:
        val_lgb
    except NameError:
        try:
            val_lgb = np.load("val_lgb_probs.npy")
        except FileNotFoundError:
            val_lgb = val_cb.copy()
            print("val_lgb не найден, дублируем val_cb.")

if y_val is not None:
    print("\n [2/3] Запуск SLSQP Оптимизатора")
    
    # Предрассчет рангов
    r_cb   = np.power(rankdata(val_cb) / len(val_cb), POWER)
    r_bert = np.power(rankdata(val_bert) / len(val_bert), POWER)
    r_lgb  = np.power(rankdata(val_lgb) / len(val_lgb), POWER)

    def objective(weights):
        w1, w2, w3 = weights
        blend = w1 * r_cb + w2 * r_bert + w3 * r_lgb
        score = average_precision_score(y_val, blend)
        return -score

    init_weights = [0.50, 0.35, 0.15]
    bounds = [(0, 1), (0, 1), (0, 1)]
    constraints = {'type': 'eq', 'fun': lambda w: 1 - sum(w)}

    res = minimize(
        objective, 
        init_weights, 
        method='SLSQP', 
        bounds=bounds, 
        constraints=constraints
    )

    best_w_cb, best_w_bert, best_w_lgb = res.x

    print("РЕЗУЛЬТАТЫ ОПТИМИЗАЦИИ ВЕСОВ")
    print(f"Оптимальный вес CatBoost Stacked : {best_w_cb:.4f}")
    print(f"Оптимальный вес BERT              : {best_w_bert:.4f}")
    print(f"Оптимальный вес LightGBM          : {best_w_lgb:.4f}")
    print(f"Максимальный PR-AUC на валидации  : {-res.fun:.5f}")
else:
    best_w_cb, best_w_bert, best_w_lgb = 0.45, 0.40, 0.15

print("\n [3/3] Генерация итогового тестового сабмита")

df_bert = pd.read_csv("/kaggle/input/datasets/liaawies/bert-fix/submission_bert_fixed.csv")
df_lgb  = pd.read_csv("/kaggle/input/datasets/liaawies/sub-lgb/submission_lgb_only.csv")
df_cb   = pd.read_csv("/kaggle/input/datasets/liaawies/sub-stuck/submission_cb_stacked.csv")

test_r_cb   = np.power(rankdata(df_cb["target"]) / len(df_cb), POWER)
test_r_bert = np.power(rankdata(df_bert["target"]) / len(df_bert), POWER)
test_r_lgb  = np.power(rankdata(df_lgb["target"]) / len(df_lgb), POWER)

test_blend = (best_w_cb * test_r_cb) + (best_w_bert * test_r_bert) + (best_w_lgb * test_r_lgb)

final_target = rankdata(test_blend) / len(test_blend)

sub_final = pd.DataFrame({
    "row_id": df_bert["row_id"],
    "target": final_target
})

output_path = "submission_slsqp_power_rank.csv"
sub_final.to_csv(output_path, index=False)

print(f" Файл {output_path} успешно сформирован!")
print(f"Размерность: {sub_final.shape}")
print(sub_final.head())

[1/3] Поиск валидационных данных
Валидационные данные не найдены. Будет использован фоллбэк с предустановленными весами

 [3/3] Генерация итогового тестового сабмита
 Файл submission_slsqp_power_rank.csv успешно сформирован!
Размерность: (11000, 2)
   row_id    target
0       0  0.194545
1       1  0.539091
2       2  0.895455
3       3  0.855545
4       4  0.084000


In [14]:
print(" [1/4] Проверка валидационных данных")

try:
    y_val = y_val.values if hasattr(y_val, 'values') else y_val
    val_cb = y_val_proba if 'y_val_proba' in locals() else val_cb
    print("✓ y_val и val_cb найдены в памяти!")
except NameError:
    try:
        y_val = np.load("y_val.npy")
        val_cb = np.load("y_val_proba_cb_stacked.npy")
        print("y_val и val_cb загружены из .npy файлов")
    except FileNotFoundError:
        y_val, val_cb = None, None
        print("Валидационные данные не найдены. Использование фоллбэка")

if y_val is not None:
    try:
        val_bert
    except NameError:
        try:
            val_bert = np.load("val_bert_probs.npy")
        except FileNotFoundError:
            val_bert = val_cb.copy()

    try:
        val_lgb
    except NameError:
        try:
            val_lgb = np.load("val_lgb_probs.npy")
        except FileNotFoundError:
            val_lgb = val_cb.copy()

if y_val is not None:
    print("\n [2/4] Подбор лучшего Power (p) через Grid Search")
    
    best_p = 1.5
    best_auc = 0.0

    for p in np.linspace(1.0, 2.5, 16):
        r_cb   = np.power(rankdata(val_cb) / len(val_cb), p)
        r_bert = np.power(rankdata(val_bert) / len(val_bert), p)
        r_lgb  = np.power(rankdata(val_lgb) / len(val_lgb), p)
        
        blend = 0.45 * r_cb + 0.40 * r_bert + 0.15 * r_lgb
        score = average_precision_score(y_val, blend)
        
        if score > best_auc:
            best_auc = score
            best_p = p

    print(f"Оптимальный Power p: {best_p:.2f} (Предварительный PR-AUC: {best_auc:.5f})")

    print("\n [3/4] Точная настройка весов SLSQP")

    r_cb   = np.power(rankdata(val_cb) / len(val_cb), best_p)
    r_bert = np.power(rankdata(val_bert) / len(val_bert), best_p)
    r_lgb  = np.power(rankdata(val_lgb) / len(val_lgb), best_p)

    def objective(weights):
        w1, w2, w3 = weights
        blend = w1 * r_cb + w2 * r_bert + w3 * r_lgb
        return -average_precision_score(y_val, blend)

    init_weights = [0.45, 0.40, 0.15]
    bounds = [(0, 1), (0, 1), (0, 1)]
    constraints = {'type': 'eq', 'fun': lambda w: 1 - sum(w)}

    res = minimize(
        objective, 
        init_weights, 
        method='SLSQP', 
        bounds=bounds, 
        constraints=constraints
    )

    best_w_cb, best_w_bert, best_w_lgb = res.x
    print("ИТОГОВЫЕ ОПТИМАЛЬНЫЕ ПАРАМЕТРЫ")
    print(f"Power (p)                        : {best_p:.2f}")
    print(f"Оптимальный вес CatBoost Stacked : {best_w_cb:.4f}")
    print(f"Оптимальный вес BERT             : {best_w_bert:.4f}")
    print(f"Оптимальный вес LightGBM         : {best_w_lgb:.4f}")
    print(f"Максимальный PR-AUC на валидации : {-res.fun:.5f}")

else:
    best_p = 1.6
    best_w_cb, best_w_bert, best_w_lgb = 0.45, 0.40, 0.15

print("\n [4/4] Формирование итогового submission")

df_bert = pd.read_csv("/kaggle/input/datasets/liaawies/bert-fix/submission_bert_fixed.csv")
df_lgb  = pd.read_csv("/kaggle/input/datasets/liaawies/sub-lgb/submission_lgb_only.csv")
df_cb   = pd.read_csv("/kaggle/input/datasets/liaawies/sub-stuck/submission_cb_stacked.csv")

test_r_cb   = np.power(rankdata(df_cb["target"]) / len(df_cb), best_p)
test_r_bert = np.power(rankdata(df_bert["target"]) / len(df_bert), best_p)
test_r_lgb  = np.power(rankdata(df_lgb["target"]) / len(df_lgb), best_p)

test_blend = (best_w_cb * test_r_cb) + (best_w_bert * test_r_bert) + (best_w_lgb * test_r_lgb)

final_target = rankdata(test_blend) / len(test_blend)

sub_final = pd.DataFrame({
    "row_id": df_bert["row_id"],
    "target": final_target
})

output_file = "submission_best_power_slsqp.csv"
sub_final.to_csv(output_file, index=False)

print(f"Файл {output_file} сформирован")
print(f"Размерность: {sub_final.shape}")
print(sub_final.head())

 [1/4] Проверка валидационных данных
✓ y_val и val_cb найдены в памяти!

 [4/4] Формирование итогового submission
Файл submission_best_power_slsqp.csv сформирован
Размерность: (11000, 2)
   row_id    target
0       0  0.194636
1       1  0.539182
2       2  0.895545
3       3  0.854182
4       4  0.085182
